<a href="https://colab.research.google.com/github/trang1981/ELAPS/blob/main/VIB_60DE1B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# XÂY DỰNG COHORT VIB FLDC-60D HOÀN CHỈNH
#
# Điều kiện:
# 1. Có ngày bắt đầu quan hệ ngân hàng
# 2. Có đủ 60 ngày quan sát
# 3. Loại khách hàng có thẻ trong 0–60 ngày đầu
# 4. Gán nhãn mới:
#       1 = có thẻ sau ngày thứ 60
#       0 = không có thẻ đến cuối kỳ nghiên cứu
#
# Chạy toàn bộ trong một ô Google Colab
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
from pathlib import Path


# ============================================================
# 1. KHAI BÁO ĐƯỜNG DẪN
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/Fintect")

FINAL_PATH = BASE_DIR / "final_dataset_no_auto_job.csv"
CUSTOMER_PATH = BASE_DIR / "1.Data_Customer.csv"
CARD_PATH = BASE_DIR / "6.Data_Card.xlsx"

# File dữ liệu FLDC-60D hoàn chỉnh
OUTPUT_PATH = BASE_DIR / "VIB_FLDC_60D.csv"

# Danh sách khách hàng bị loại do mở thẻ sớm
EARLY_ADOPTER_PATH = (
    BASE_DIR / "FLDC60_removed_early_adopters.csv"
)

# Danh sách khách hàng không đủ 60 ngày quan sát
INSUFFICIENT_WINDOW_PATH = (
    BASE_DIR / "FLDC60_removed_insufficient_window.csv"
)

# Danh sách ngày bất thường
INVALID_DATE_PATH = (
    BASE_DIR / "FLDC60_invalid_dates.csv"
)

# Timeline toàn bộ khách hàng
TIMELINE_PATH = (
    BASE_DIR / "FLDC60_customer_timeline.csv"
)

HORIZON_DAYS = 60


# ============================================================
# 2. ĐỌC DỮ LIỆU
# ============================================================

final_df = pd.read_csv(
    FINAL_PATH,
    low_memory=False
)

customer_df = pd.read_csv(
    CUSTOMER_PATH,
    low_memory=False
)

card_df = pd.read_excel(
    CARD_PATH,
    engine="openpyxl"
)

# Chuẩn hóa tên cột
final_df.columns = final_df.columns.astype(str).str.strip()
customer_df.columns = customer_df.columns.astype(str).str.strip()
card_df.columns = card_df.columns.astype(str).str.strip()

print("=" * 72)
print("KÍCH THƯỚC DỮ LIỆU BAN ĐẦU")
print("=" * 72)

print(f"Final dataset : {final_df.shape}")
print(f"Customer      : {customer_df.shape}")
print(f"Card          : {card_df.shape}")


# ============================================================
# 3. KHAI BÁO CỘT
# ============================================================

ID_COL = "CUSTOMER_NUMBER"

RELATIONSHIP_DATE_COL = "CLIENT_CREATE_DATE"

CARD_MONTH_COL = "MONTH"

CARD_COUNT_COL = "COUNT_CREDITCARD"


# ============================================================
# 4. KIỂM TRA CỘT BẮT BUỘC
# ============================================================

required_columns = [
    (final_df, ID_COL, "final_dataset_no_auto_job.csv"),
    (customer_df, ID_COL, "1.Data_Customer.csv"),
    (
        customer_df,
        RELATIONSHIP_DATE_COL,
        "1.Data_Customer.csv"
    ),
    (card_df, ID_COL, "6.Data_Card.xlsx"),
    (card_df, CARD_MONTH_COL, "6.Data_Card.xlsx"),
    (card_df, CARD_COUNT_COL, "6.Data_Card.xlsx")
]

for df, column_name, file_name in required_columns:
    if column_name not in df.columns:
        raise KeyError(
            f"Không tìm thấy cột '{column_name}' "
            f"trong file {file_name}.\n"
            f"Các cột hiện có:\n{df.columns.tolist()}"
        )

print("\nĐã xác định đầy đủ các cột cần thiết.")


# ============================================================
# 5. CHUẨN HÓA CUSTOMER_NUMBER
# ============================================================

def normalize_customer_id(series):
    """
    Chuẩn hóa mã khách hàng giữa ba file.
    """

    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA
        })
    )


final_df["_CUSTOMER_ID"] = normalize_customer_id(
    final_df[ID_COL]
)

customer_df["_CUSTOMER_ID"] = normalize_customer_id(
    customer_df[ID_COL]
)

card_df["_CUSTOMER_ID"] = normalize_customer_id(
    card_df[ID_COL]
)


# ============================================================
# 6. CHUYỂN ĐỔI NGÀY
# ============================================================

customer_df["_RELATIONSHIP_DATE"] = pd.to_datetime(
    customer_df[RELATIONSHIP_DATE_COL],
    format="%Y-%m-%d",
    errors="coerce"
)

card_df["_CARD_MONTH"] = pd.to_datetime(
    card_df[CARD_MONTH_COL],
    errors="coerce"
)

card_df["_CARD_COUNT"] = pd.to_numeric(
    card_df[CARD_COUNT_COL],
    errors="coerce"
).fillna(0)


# ============================================================
# 7. XÁC ĐỊNH NGÀY KẾT THÚC NGHIÊN CỨU
#
# Lấy tháng lớn nhất có trong file Card.
# Với dữ liệu hiện tại dự kiến là 31/12/2019.
# ============================================================

STUDY_END_DATE = card_df["_CARD_MONTH"].max()

if pd.isna(STUDY_END_DATE):
    raise ValueError(
        "Không xác định được ngày kết thúc dữ liệu "
        "từ cột MONTH."
    )

print("\nThông tin thời gian:")
print(
    "Ngày nhỏ nhất trong Card:",
    card_df["_CARD_MONTH"].min()
)
print(
    "Ngày kết thúc nghiên cứu:",
    STUDY_END_DATE
)
print(
    "Độ dài cửa sổ quan sát:",
    HORIZON_DAYS,
    "ngày"
)


# ============================================================
# 8. CHỈ XỬ LÝ KHÁCH HÀNG TRONG FINAL DATASET
# ============================================================

final_ids = set(
    final_df["_CUSTOMER_ID"].dropna().unique()
)

customer_sub = customer_df.loc[
    customer_df["_CUSTOMER_ID"].isin(final_ids),
    [
        "_CUSTOMER_ID",
        "_RELATIONSHIP_DATE"
    ]
].copy()

card_sub = card_df.loc[
    card_df["_CUSTOMER_ID"].isin(final_ids),
    [
        "_CUSTOMER_ID",
        "_CARD_MONTH",
        "_CARD_COUNT"
    ]
].copy()


# ============================================================
# 9. LẤY NGÀY BẮT ĐẦU QUAN HỆ
#
# Nếu CUSTOMER_NUMBER xuất hiện nhiều lần,
# lấy CLIENT_CREATE_DATE sớm nhất.
# ============================================================

customer_dates = (
    customer_sub
    .dropna(subset=["_CUSTOMER_ID"])
    .groupby(
        "_CUSTOMER_ID",
        as_index=False
    )
    .agg(
        relationship_date=(
            "_RELATIONSHIP_DATE",
            "min"
        )
    )
)


# ============================================================
# 10. LẤY THỜI ĐIỂM ĐẦU TIÊN CÓ THẺ
#
# first_card_date =
# MONTH sớm nhất mà COUNT_CREDITCARD > 0
# ============================================================

positive_card_rows = card_sub.loc[
    (card_sub["_CARD_COUNT"] > 0)
    & card_sub["_CUSTOMER_ID"].notna()
    & card_sub["_CARD_MONTH"].notna()
].copy()

first_card_dates = (
    positive_card_rows
    .groupby(
        "_CUSTOMER_ID",
        as_index=False
    )
    .agg(
        first_card_date=(
            "_CARD_MONTH",
            "min"
        )
    )
)


# ============================================================
# 11. TẠO TIMELINE CHO TOÀN BỘ FINAL DATASET
#
# Bắt đầu từ final dataset để không làm mất khách hàng
# không xuất hiện trong bảng Card.
# ============================================================

timeline_df = (
    final_df[["_CUSTOMER_ID"]]
    .drop_duplicates()
    .merge(
        customer_dates,
        on="_CUSTOMER_ID",
        how="left",
        validate="one_to_one"
    )
    .merge(
        first_card_dates,
        on="_CUSTOMER_ID",
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 12. TẠO NGÀY CẮT FLDC-60D
# ============================================================

timeline_df["feature_cutoff_date"] = (
    timeline_df["relationship_date"]
    + pd.Timedelta(days=HORIZON_DAYS)
)

timeline_df["days_to_first_card"] = (
    timeline_df["first_card_date"]
    - timeline_df["relationship_date"]
).dt.days


# ============================================================
# 13. XÁC ĐỊNH CÁC ĐIỀU KIỆN LOẠI
# ============================================================

# Không xác định được ngày bắt đầu quan hệ
timeline_df["missing_relationship_date"] = (
    timeline_df["relationship_date"].isna()
)

# Không đủ 60 ngày quan sát
timeline_df["insufficient_60d_window"] = (
    timeline_df["relationship_date"].notna()
    & (
        timeline_df["feature_cutoff_date"]
        > STUDY_END_DATE
    )
)

# Ngày thẻ trước ngày bắt đầu quan hệ
timeline_df["invalid_card_date"] = (
    timeline_df["first_card_date"].notna()
    & timeline_df["relationship_date"].notna()
    & (
        timeline_df["first_card_date"]
        < timeline_df["relationship_date"]
    )
)

# Có thẻ trong 0–60 ngày đầu
timeline_df["early_adopter_60d"] = (
    timeline_df["first_card_date"].notna()
    & timeline_df["relationship_date"].notna()
    & (
        timeline_df["first_card_date"]
        >= timeline_df["relationship_date"]
    )
    & (
        timeline_df["first_card_date"]
        <= timeline_df["feature_cutoff_date"]
    )
)


# ============================================================
# 14. XÁC ĐỊNH KHÁCH HÀNG ĐƯỢC GIỮ TRONG FLDC-60D
#
# Chỉ giữ khách hàng:
# - có ngày bắt đầu quan hệ;
# - đủ 60 ngày quan sát;
# - ngày dữ liệu hợp lệ;
# - không mở thẻ trong 60 ngày đầu.
# ============================================================

timeline_df["eligible_fldc60"] = (
    ~timeline_df["missing_relationship_date"]
    & ~timeline_df["insufficient_60d_window"]
    & ~timeline_df["invalid_card_date"]
    & ~timeline_df["early_adopter_60d"]
)


# ============================================================
# 15. GÁN NHÃN STRICTLY-FUTURE
#
# Label = 1 khi:
# - có thẻ;
# - ngày có thẻ sau cutoff 60 ngày;
# - ngày có thẻ không vượt quá ngày kết thúc nghiên cứu.
#
# Label = 0 khi:
# - không có thẻ đến cuối kỳ nghiên cứu.
# ============================================================

timeline_df["TARGET_FLDC_60D"] = 0

positive_mask = (
    timeline_df["eligible_fldc60"]
    & timeline_df["first_card_date"].notna()
    & (
        timeline_df["first_card_date"]
        > timeline_df["feature_cutoff_date"]
    )
    & (
        timeline_df["first_card_date"]
        <= STUDY_END_DATE
    )
)

timeline_df.loc[
    positive_mask,
    "TARGET_FLDC_60D"
] = 1


# ============================================================
# 16. GÁN LÝ DO LOẠI
# ============================================================

timeline_df["exclusion_reason"] = "INCLUDED_FLDC60"

timeline_df.loc[
    timeline_df["missing_relationship_date"],
    "exclusion_reason"
] = "MISSING_RELATIONSHIP_DATE"

timeline_df.loc[
    timeline_df["insufficient_60d_window"],
    "exclusion_reason"
] = "INSUFFICIENT_60D_WINDOW"

timeline_df.loc[
    timeline_df["invalid_card_date"],
    "exclusion_reason"
] = "INVALID_CARD_BEFORE_RELATIONSHIP"

timeline_df.loc[
    timeline_df["early_adopter_60d"],
    "exclusion_reason"
] = "EARLY_ADOPTER_WITHIN_60D"


# ============================================================
# 17. TẠO COHORT FLDC-60D
# ============================================================

fldc_timeline = timeline_df.loc[
    timeline_df["eligible_fldc60"]
].copy()

fldc_ids = set(
    fldc_timeline["_CUSTOMER_ID"].dropna()
)

fldc_df = final_df.loc[
    final_df["_CUSTOMER_ID"].isin(fldc_ids)
].copy()


# ============================================================
# 18. XÓA NHÃN CŨ VÀ GHÉP NHÃN FLDC-60D MỚI
# ============================================================

# COUNT_CREDITCARD trong final dataset là nhãn cũ.
# Xóa để tránh nhầm với nhãn strictly-future.
fldc_df.drop(
    columns=["COUNT_CREDITCARD"],
    inplace=True,
    errors="ignore"
)

fldc_df = fldc_df.merge(
    fldc_timeline[
        [
            "_CUSTOMER_ID",
            "relationship_date",
            "feature_cutoff_date",
            "first_card_date",
            "days_to_first_card",
            "TARGET_FLDC_60D"
        ]
    ],
    on="_CUSTOMER_ID",
    how="inner",
    validate="one_to_one"
)

# Đổi nhãn mới thành COUNT_CREDITCARD
# để tương thích với code huấn luyện hiện tại.
fldc_df.rename(
    columns={
        "TARGET_FLDC_60D": "COUNT_CREDITCARD"
    },
    inplace=True
)


# ============================================================
# 19. TẠO CÁC BẢNG KHÁCH HÀNG BỊ LOẠI
# ============================================================

early_adopter_df = timeline_df.loc[
    timeline_df["early_adopter_60d"]
].copy()

insufficient_window_df = timeline_df.loc[
    timeline_df["insufficient_60d_window"]
].copy()

invalid_date_df = timeline_df.loc[
    timeline_df["invalid_card_date"]
].copy()


# ============================================================
# 20. THỐNG KÊ SỐ LƯỢNG
# ============================================================

n_initial = len(final_df)

n_missing_relationship = int(
    timeline_df["missing_relationship_date"].sum()
)

n_insufficient = int(
    timeline_df["insufficient_60d_window"].sum()
)

n_invalid = int(
    timeline_df["invalid_card_date"].sum()
)

# Early adopters trong nhóm có đủ cửa sổ và ngày hợp lệ
n_early_eligible_window = int(
    (
        timeline_df["early_adopter_60d"]
        & ~timeline_df["insufficient_60d_window"]
        & ~timeline_df["missing_relationship_date"]
        & ~timeline_df["invalid_card_date"]
    ).sum()
)

n_final = len(fldc_df)

n_positive = int(
    fldc_df["COUNT_CREDITCARD"].sum()
)

n_negative = int(
    (fldc_df["COUNT_CREDITCARD"] == 0).sum()
)

positive_rate = (
    n_positive / n_final
    if n_final > 0
    else 0
)


# ============================================================
# 21. KIỂM TRA TÍNH NHẤT QUÁN
# ============================================================

assert len(fldc_df) == fldc_df[ID_COL].nunique(), (
    "Cohort FLDC-60D có CUSTOMER_NUMBER trùng lặp."
)

assert fldc_df["COUNT_CREDITCARD"].isin([0, 1]).all(), (
    "Nhãn FLDC-60D không chỉ gồm 0 và 1."
)

assert (
    fldc_df["feature_cutoff_date"]
    <= STUDY_END_DATE
).all(), (
    "Vẫn còn khách hàng không đủ 60 ngày quan sát."
)

assert not (
    fldc_df["first_card_date"].notna()
    & (
        fldc_df["first_card_date"]
        <= fldc_df["feature_cutoff_date"]
    )
).any(), (
    "Vẫn còn early adopter trong cohort FLDC-60D."
)

assert n_positive + n_negative == n_final, (
    "Tổng số nhãn 0 và nhãn 1 không bằng kích thước cohort."
)


# ============================================================
# 22. XÓA CỘT KỸ THUẬT TRƯỚC KHI LƯU
# ============================================================

fldc_df.drop(
    columns=["_CUSTOMER_ID"],
    inplace=True,
    errors="ignore"
)


# ============================================================
# 23. XUẤT FILE
# ============================================================

fldc_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

early_adopter_df.to_csv(
    EARLY_ADOPTER_PATH,
    index=False,
    encoding="utf-8-sig"
)

insufficient_window_df.to_csv(
    INSUFFICIENT_WINDOW_PATH,
    index=False,
    encoding="utf-8-sig"
)

invalid_date_df.to_csv(
    INVALID_DATE_PATH,
    index=False,
    encoding="utf-8-sig"
)

timeline_df.to_csv(
    TIMELINE_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 24. IN THỐNG KÊ CUỐI CÙNG
# ============================================================

print("\n" + "=" * 72)
print("KẾT QUẢ XÂY DỰNG VIB FLDC-60D")
print("=" * 72)

print(f"Ngày kết thúc nghiên cứu              : {STUDY_END_DATE.date()}")
print(f"Độ dài cửa sổ feature                 : {HORIZON_DAYS} ngày")

print("\n--- QUÁ TRÌNH LỌC ---")

print(f"Số khách hàng ban đầu                 : {n_initial:,}")

print(
    f"Thiếu ngày bắt đầu quan hệ            : "
    f"{n_missing_relationship:,}"
)

print(
    f"Không đủ 60 ngày quan sát             : "
    f"{n_insufficient:,}"
)

print(
    f"Ngày thẻ trước ngày quan hệ           : "
    f"{n_invalid:,}"
)

print(
    f"Mở thẻ trong 60 ngày đầu bị loại      : "
    f"{n_early_eligible_window:,}"
)

print(
    f"Customers after removal               : "
    f"{n_final:,}"
)

print("\n--- PHÂN BỐ NHÃN FLDC-60D ---")

print(
    f"Negative customers – nhãn 0           : "
    f"{n_negative:,}"
)

print(
    f"Positive customers – nhãn 1           : "
    f"{n_positive:,}"
)

print(
    f"Positive rate                          : "
    f"{positive_rate:.4%}"
)

print("\nBảng tần số nhãn:")

label_statistics = (
    fldc_df["COUNT_CREDITCARD"]
    .value_counts()
    .sort_index()
    .rename_axis("Label")
    .reset_index(name="Customers")
)

label_statistics["Percentage"] = (
    label_statistics["Customers"]
    / n_final
    * 100
)

print(label_statistics.to_string(index=False))

print("\n--- FILE ĐÃ TẠO ---")

print(f"1. Cohort FLDC-60D:\n   {OUTPUT_PATH}")

print(
    "2. Khách hàng mở thẻ trong 60 ngày đầu:\n"
    f"   {EARLY_ADOPTER_PATH}"
)

print(
    "3. Khách hàng không đủ 60 ngày quan sát:\n"
    f"   {INSUFFICIENT_WINDOW_PATH}"
)

print(
    "4. Trường hợp ngày bất thường:\n"
    f"   {INVALID_DATE_PATH}"
)

print(
    "5. Timeline toàn bộ khách hàng:\n"
    f"   {TIMELINE_PATH}"
)

print("\nKiểm tra đạt:")
print("- Mọi khách hàng đều có đủ 60 ngày quan sát.")
print("- Không còn khách hàng mở thẻ trong 60 ngày đầu.")
print("- Nhãn 1 chỉ phản ánh mở thẻ strictly after day 60.")
print("- Tổng nhãn 0 + nhãn 1 bằng tổng kích thước cohort.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
KÍCH THƯỚC DỮ LIỆU BAN ĐẦU
Final dataset : (127460, 20)
Customer      : (290223, 9)
Card          : (871589, 4)

Đã xác định đầy đủ các cột cần thiết.

Thông tin thời gian:
Ngày nhỏ nhất trong Card: 2019-01-31 00:00:00
Ngày kết thúc nghiên cứu: 2019-12-31 00:00:00
Độ dài cửa sổ quan sát: 60 ngày

KẾT QUẢ XÂY DỰNG VIB FLDC-60D
Ngày kết thúc nghiên cứu              : 2019-12-31
Độ dài cửa sổ feature                 : 60 ngày

--- QUÁ TRÌNH LỌC ---
Số khách hàng ban đầu                 : 127,460
Thiếu ngày bắt đầu quan hệ            : 0
Không đủ 60 ngày quan sát             : 18,681
Ngày thẻ trước ngày quan hệ           : 0
Mở thẻ trong 60 ngày đầu bị loại      : 14,326
Customers after removal               : 94,453

--- PHÂN BỐ NHÃN FLDC-60D ---
Negative customers – nhãn 0           : 90,079
Positive customers – nhãn 1           : 4,374
Positive rate           